In [1]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EvalPrediction,
    EarlyStoppingCallback,
    pipeline
)
import evaluate

# ==========================================
# 1. CONFIGURAÇÕES INICIAIS
# ==========================================
df = pd.read_csv("df_exportado.csv")
model_name = "answerdotai/ModernBERT-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"A usar dispositivo: {device}")

label_map = {'Human': 0, 'OpenAI': 1, 'Google': 2, 'Meta': 3, 'Anthropic': 4}
id_to_label = {v: k for k, v in label_map.items()}

c:\Users\Carlos\Documents\ProjetoAP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


A usar dispositivo: cuda


In [ ]:
# ==========================================
# 2. PREPARAÇÃO, LIMPEZA E UNDERSAMPLING DO DATASET
# ==========================================
# Assume-se que o teu dataset original de treino está carregado na variável 'df'
df['labels'] = df['Label'].map(label_map) # Usar 'labels' com 's'
df['Text'] = df['Text'].fillna("").astype(str)
df = df[df['Text'].str.strip() != ""].copy()

# Aumentámos para 5000 para dar mais dados ao modelo e evitar Underfitting
AMOSTRAS_POR_CLASSE = 5000 
amostras_list = []

for classe_id in df['labels'].dropna().unique():
    df_filtro = df[df['labels'] == classe_id]
    n_escolher = min(len(df_filtro), AMOSTRAS_POR_CLASSE)
    amostras_list.append(df_filtro.sample(n=n_escolher, random_state=42))

df_reduzido = pd.concat(amostras_list).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Dataset reduzido e balanceado! Total de linhas de treino/validação: {len(df_reduzido)}")
# Criar o Dataset da Hugging Face
dataset = Dataset.from_pandas(df_reduzido[['Text', 'labels']])
dataset = dataset.train_test_split(test_size=0.2, seed=42)
print(df_reduzido['labels'].value_counts())

Dataset reduzido e balanceado! Total de linhas de treino/validação: 25000
labels
3    5000
4    5000
2    5000
1    5000
0    5000
Name: count, dtype: int64


In [7]:
# ==========================================
# 3. TOKENIZAÇÃO
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    textos = [str(t) for t in examples["Text"]]
    # Aumentámos para 512 para ele ler o texto quase todo e detetar o Google/Meta
    return tokenizer(textos, padding="max_length", truncation=True, max_length=512)

print("A tokenizar os dados...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

A tokenizar os dados...


Map: 100%|██████████| 5000/5000 [00:01<00:00, 3142.57 examples/s]


In [8]:
# ==========================================
# 4. MODELO E MÉTRICAS
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=5,
    id2label=id_to_label,
    label2id=label_map
).to(device)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred: EvalPrediction) -> dict:
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.argmax(predictions, axis=1)
    resultado = accuracy_metric.compute(predictions=predictions, references=labels)
    return resultado if resultado is not None else {}

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 2856.13it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# ==========================================
# 5. TRAINER
# ==========================================
training_args = TrainingArguments(
    output_dir="./results_modernbert",
    learning_rate=2e-5,          
    per_device_train_batch_size=16, # Atenção: se der "Out of Memory", baixar para 8
    per_device_eval_batch_size=16,  # Atenção: se der "Out of Memory", baixar para 8
    num_train_epochs=5,             # Aumentámos para 5
    weight_decay=0.01,
    eval_strategy="epoch",       
    save_strategy="epoch",
    load_best_model_at_end=True,   
    metric_for_best_model="accuracy",
    greater_is_better=True,
    push_to_hub=False,
    fp16=True if torch.cuda.is_available() else False,
    save_total_limit=2             
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],   # type: ignore
    eval_dataset=tokenized_datasets["test"],     # type: ignore
    processing_class=tokenizer,                  # type: ignore
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("A iniciar o Treino do ModernBERT...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


A iniciar o Treino do ModernBERT...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.020680,0.009526,0.997000
2,0.001065,0.011685,0.997800
3,0.001305,0.020069,0.995800
4,0.000605,0.007401,0.999200
5,0.000001,0.006790,0.999200


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]


TrainOutput(global_step=6250, training_loss=0.013203131213188172, metrics={'train_runtime': 2674.2329, 'train_samples_per_second': 37.394, 'train_steps_per_second': 2.337, 'total_flos': 3.4076493312e+16, 'train_loss': 0.013203131213188172, 'epoch': 5.0})

In [11]:
# ==========================================
# 6. EXPORTAR MODELO
# ==========================================
print("\nA guardar o modelo...")
caminho_guardar = "Subm2/modernbert_final"
trainer.save_model(caminho_guardar)
tokenizer.save_pretrained(caminho_guardar)


A guardar o modelo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


('Subm2/modernbert_final\\tokenizer_config.json',
 'Subm2/modernbert_final\\tokenizer.json')

In [ ]:
from huggingface_hub import login

login("...")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

caminho_modelo = "./Subm2/modernbert_final"
modelo_local = AutoModelForSequenceClassification.from_pretrained(caminho_modelo)
tokenizer_local = AutoTokenizer.from_pretrained(caminho_modelo)

nome_no_hub = "carlosf21/projeto-ap-modernbert"

print(f"A fazer upload para {nome_no_hub}... Isto pode demorar uns minutos (são ~600MB)")

modelo_local.push_to_hub(nome_no_hub)
tokenizer_local.push_to_hub(nome_no_hub)

print("Upload concluído com sucesso!")

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 6430.41it/s]


A fazer upload para carlosf21/projeto-ap-modernbert... Isto pode demorar uns minutos (são ~600MB) 🚀


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]
Processing Files (1 / 1): 100%|██████████|  598MB /  598MB, 5.27MB/s  
New Data Upload: 100%|██████████|  598MB /  598MB, 5.27MB/s  
c:\Users\Carlos\Documents\ProjetoAP\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Carlos\.cache\huggingface\hub\models--carlosf21--projeto-ap-modernbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://doc

✅ Upload concluído com sucesso! O teu modelo já está online.
